In [ ]:
import pandas as pd

historico = pd.read_csv("/content/Base_Historica_PIB_Ecuador_Unidad5.csv")

print(historico.shape)
historico.head()

In [ ]:
diccionario = pd.read_csv(
    "/content/Diccionario_Datos_PIB_Ecuador_Unidad5.csv"
)

# Revisar las variables y sus definiciones
display(diccionario)

Seleccionar las variables predictoras

In [ ]:
# Variable objetivo
TARGET = "crecimiento_pib"

# PERSONALIZACIÓN DEL ESTUDIANTE: cambiar SOLO esta lista.
# Elegir entre 4 y 8 indicadores numéricos existentes en el CSV.
FEATURES = [
    "precio_petroleo",
    "credito_privado",
    "ventas_internas",
    "consumo_electrico"
]

if not 4 <= len(FEATURES) <= 8:
    raise ValueError("Seleccione entre 4 y 8 predictores")
if len(FEATURES) != len(set(FEATURES)):
    raise ValueError("No repita predictores")
faltantes = [v for v in FEATURES + [TARGET, "trimestre"] if v not in historico.columns]
if faltantes:
    raise ValueError(f"Columnas inexistentes: {faltantes}")
no_numericas = [v for v in FEATURES if not pd.api.types.is_numeric_dtype(historico[v])]
if no_numericas:
    raise ValueError(f"Estas variables no son numéricas: {no_numericas}")

print("Variable objetivo:", TARGET)
print("Predictores seleccionados:", FEATURES)


In [ ]:
# X contiene las variables predictoras
X = historico[FEATURES]

# y contiene la variable objetivo
y = historico[TARGET]

# Verificamos las dimensiones
print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

# Visualizamos los predictores
display(X.head())

Separar entrenamiento de prueba

In [ ]:
# Ordenamos la base cronológicamente
historico = historico.sort_values("trimestre").reset_index(drop=True)

# Volvemos a crear X e y después de ordenar los trimestres
X = historico[FEATURES]
y = historico[TARGET]

# Entrenamiento: 2005Q1 hasta 2024Q4
train = historico[
    historico["trimestre"] <= "2024Q4"
].copy()

# Prueba: 2025Q1 hasta 2026Q3
test = historico[
    historico["trimestre"] >= "2025Q1"
].copy()

# Separamos las variables
X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

# Comprobamos las particiones
print("Entrenamiento:", len(train))
print("Prueba:", len(test))

print("Primero en prueba:", test["trimestre"].iloc[0])
print("Último en prueba:", test["trimestre"].iloc[-1])

Construir Preprocesamiento

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Definimos la preparación de variables
preprocesador = ColumnTransformer(
    transformers=[
        (
            "numericas",
            SimpleImputer(strategy="median"),
            FEATURES
        )
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

pipeline_rf = Pipeline(steps=[
    (
        "preprocesamiento",
        preprocesador
    ),
    (
        "modelo",
        RandomForestRegressor(
            n_estimators=180,
            max_depth=5,
            min_samples_leaf=3,
            max_features=1.0,
            random_state=2026,
            n_jobs=-1
        )
    )
])

print(pipeline_rf)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate

# Definimos la validación cruzada temporal
validacion = TimeSeriesSplit(
    n_splits=4,
    test_size=4
)

# Revisamos los períodos de cada partición
for numero, (indices_train, indices_valid) in enumerate(
    validacion.split(X_train), start=1
):
    print("Validación:", numero)

    print(
        "Entrenamiento:",
        train.iloc[indices_train]["trimestre"].iloc[0],
        "hasta",
        train.iloc[indices_train]["trimestre"].iloc[-1]
    )

    print(
        "Validación:",
        train.iloc[indices_valid]["trimestre"].iloc[0],
        "hasta",
        train.iloc[indices_valid]["trimestre"].iloc[-1]
    )

    print("-------------------------")

Ejecutamos validación cruzada

In [ ]:
# Definimos las métricas de evaluación
metricas = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error"
}

# Ejecutamos la validación cruzada
resultados_cv = cross_validate(
    pipeline_rf,
    X_train,
    y_train,
    cv=validacion,
    scoring=metricas
)

# Convertimos los resultados en una tabla
tabla_cv = pd.DataFrame({
    "Validacion": [1, 2, 3, 4],
    "MAE": -resultados_cv["test_MAE"],
    "RMSE": -resultados_cv["test_RMSE"]
})

display(tabla_cv)

In [ ]:
print("MAE promedio:", round(tabla_cv["MAE"].mean(), 3))

print("RMSE promedio:", round(tabla_cv["RMSE"].mean(), 3))

Entrenar el modelo y evaluar la muestra final

In [ ]:
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    root_mean_squared_error, r2_score
)

# Entrenamiento con 2005Q1–2024Q4
pipeline_rf.fit(X_train, y_train)

# Predicción de los 7 trimestres reservados
predicciones_test = pipeline_rf.predict(X_test)

# Métricas de prueba
mae_test = mean_absolute_error(y_test, predicciones_test)
mse_test = mean_squared_error(y_test, predicciones_test)
rmse_test = root_mean_squared_error(y_test, predicciones_test)
r2_test = r2_score(y_test, predicciones_test)

print("MAE prueba:", round(mae_test, 3))
print("MSE prueba:", round(mse_test, 3))
print("RMSE prueba:", round(rmse_test, 3))
print("R² prueba:", round(r2_test, 3))

# Creamos el DataFrame de resultados para mostrar
resultados_test = pd.DataFrame({
    "Trimestre": test["trimestre"],
    "PIB_real": y_test,
    "PIB_predicho": predicciones_test,
    "Residuo": y_test - predicciones_test
})

display(resultados_test)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
    r2_score
)

# MAE
mae_test = mean_absolute_error(
    y_test, predicciones_test
)

# MSE
mse_test = mean_squared_error(
    y_test, predicciones_test
)

# RMSE
rmse_test = root_mean_squared_error(
    y_test, predicciones_test
)

# R cuadrado
r2_test = r2_score(
    y_test, predicciones_test
)

print("MAE prueba:", round(mae_test, 3))
print("MSE prueba:", round(mse_test, 3))
print("RMSE prueba:", round(rmse_test, 3))
print("R² prueba:", round(r2_test, 3))

In [ ]:
print("Observaciones en prueba:", len(X_test))
print("Primer trimestre:", test["trimestre"].iloc[0])
print("Último trimestre:", test["trimestre"].iloc[-1])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    resultados_test["Trimestre"],
    resultados_test["PIB_real"],
    marker="o",
    label="PIB observado"
)

plt.plot(
    resultados_test["Trimestre"],
    resultados_test["PIB_predicho"],
    marker="s",
    label="PIB predicho"
)

plt.axhline(0, color="gray", linestyle="--")

plt.title("Ecuador: crecimiento del PIB observado y predicho")
plt.xlabel("Trimestre")
plt.ylabel("Crecimiento interanual del PIB (%)")

plt.xticks(rotation=45)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Experiment Tracker — seguimiento de ejecuciones (sin MLflow)

En esta sección probamos **ocho configuraciones** de Random Forest sobre **las mismas cuatro ventanas de validación cruzada temporal**. Cada configuración se identifica por un `run_id` y sus métricas se guardan en archivos CSV para la aplicación Streamlit.

**Importante:** la selección se basa únicamente en la validación cruzada del conjunto de entrenamiento (hasta 2024Q4). El período 2025Q1–2026Q3 se conserva como prueba independiente y se evalúa **una sola vez después de seleccionar el ganador**. Los resultados no son simulados por este bloque: se calculan al ejecutar el notebook con la base histórica cargada.


In [ ]:
from datetime import datetime, timezone
from itertools import product
from sklearn.base import clone
from sklearn.model_selection import cross_validate

# Cuatro particiones temporales de cuatro trimestres, como en el ejercicio original
cv_tracker = TimeSeriesSplit(n_splits=4, test_size=4)

# Cuadrícula pequeña y didáctica: 2 × 2 × 2 = 8 experimentos
combinaciones = list(product([100, 200], [4, 7], [2, 4]))
registros_runs = []
registros_folds = []
fecha_ejecucion = datetime.now(timezone.utc).isoformat(timespec="seconds")

for numero, (arboles, profundidad, hoja) in enumerate(combinaciones, start=1):
    run_id = f"RUN-{numero:03d}"
    candidato = clone(pipeline_rf)
    candidato.set_params(
        modelo__n_estimators=arboles,
        modelo__max_depth=profundidad,
        modelo__min_samples_leaf=hoja
    )
    puntajes = cross_validate(
        candidato, X_train, y_train, cv=cv_tracker,
        scoring={"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error"},
        error_score="raise"
    )
    maes = -puntajes["test_MAE"]
    rmses = -puntajes["test_RMSE"]
    registros_runs.append({
        "run_id": run_id, "fecha_utc": fecha_ejecucion,
        "modelo": "Random Forest", "n_estimators": arboles,
        "max_depth": profundidad, "min_samples_leaf": hoja,
        "max_features": 1.0, "random_state": 2026,
        "n_splits": 4, "test_size": 4,
        "periodo_entrenamiento_fin": str(train["trimestre"].iloc[-1]),
        "n_variables": len(FEATURES), "variables": ", ".join(FEATURES),
        "mae_cv": float(maes.mean()), "rmse_cv": float(rmses.mean()),
        "std_rmse_cv": float(rmses.std(ddof=1)),
    })
    for fold, (mae, rmse) in enumerate(zip(maes, rmses), start=1):
        indices_train, indices_valid = list(cv_tracker.split(X_train))[fold-1]
        registros_folds.append({
            "run_id": run_id, "fold": fold,
            "fin_entrenamiento": str(train.iloc[indices_train]["trimestre"].iloc[-1]),
            "inicio_validacion": str(train.iloc[indices_valid]["trimestre"].iloc[0]),
            "fin_validacion": str(train.iloc[indices_valid]["trimestre"].iloc[-1]),
            "mae": float(mae), "rmse": float(rmse)
        })

experimentos = pd.DataFrame(registros_runs).sort_values(["rmse_cv", "mae_cv"]).reset_index(drop=True)
experimentos["ranking"] = range(1, len(experimentos)+1)
experimentos["es_mejor"] = experimentos["ranking"] == 1
folds_experimentos = pd.DataFrame(registros_folds)

display(experimentos[["run_id", "n_estimators", "max_depth", "min_samples_leaf", "mae_cv", "rmse_cv", "ranking"]])


In [ ]:
# Seleccionamos el mejor modelo usando ÚNICAMENTE validación cruzada
mejor = experimentos.iloc[0]
mejor_run_id = mejor["run_id"]
modelo_ganador = clone(pipeline_rf).set_params(
    modelo__n_estimators=int(mejor["n_estimators"]),
    modelo__max_depth=int(mejor["max_depth"]),
    modelo__min_samples_leaf=int(mejor["min_samples_leaf"])
)

# Evaluación final del ganador en el período de prueba reservado
modelo_ganador.fit(X_train, y_train)
predicciones_ganador = modelo_ganador.predict(X_test)
metricas_ganador_test = {
    "run_id": mejor_run_id,
    "mae_test": float(mean_absolute_error(y_test, predicciones_ganador)),
    "rmse_test": float(root_mean_squared_error(y_test, predicciones_ganador)),
    "r2_test": float(r2_score(y_test, predicciones_ganador)),
    "n_test": len(X_test),
    "inicio_test": str(test["trimestre"].iloc[0]),
    "fin_test": str(test["trimestre"].iloc[-1]),
}

# No comparamos múltiples modelos con el test para volver a elegir.
print("Run seleccionado por validación cruzada:", mejor_run_id)
print("RMSE CV:", round(float(mejor["rmse_cv"]), 3))
print("RMSE prueba independiente:", round(metricas_ganador_test["rmse_test"], 3))

test_ganador = pd.DataFrame({
    "trimestre": test["trimestre"].to_numpy(),
    "pib_observado": y_test.to_numpy(),
    "pib_predicho": predicciones_ganador,
})
test_ganador["error"] = test_ganador["pib_observado"] - test_ganador["pib_predicho"]


In [ ]:
# Archivos que alimentarán el nuevo panel de Streamlit
from pathlib import Path
import json

carpeta_salidas = Path("salidas_tracker")
carpeta_salidas.mkdir(exist_ok=True)
experimentos.to_csv(carpeta_salidas / "experimentos.csv", index=False)
folds_experimentos.to_csv(carpeta_salidas / "experimentos_folds.csv", index=False)
test_ganador.to_csv(carpeta_salidas / "predicciones_test_ganador.csv", index=False)
with open(carpeta_salidas / "mejor_experimento.json", "w", encoding="utf-8") as archivo:
    json.dump({
        "run_id": mejor_run_id,
        "parametros": {
            "n_estimators": int(mejor["n_estimators"]),
            "max_depth": int(mejor["max_depth"]),
            "min_samples_leaf": int(mejor["min_samples_leaf"])
        },
        "metricas_cv": {
            "mae": float(mejor["mae_cv"]),
            "rmse": float(mejor["rmse_cv"])
        },
        "metricas_test": metricas_ganador_test,
        "features": FEATURES
    }, archivo, indent=2, ensure_ascii=False)

print("Archivos disponibles:")
for archivo in carpeta_salidas.iterdir():
    print("-", archivo)

# Descargar todos los resultados juntos desde Colab (opcional)
# !zip -r salidas_tracker.zip salidas_tracker
# from google.colab import files
# files.download("salidas_tracker.zip")

# Configuración que Streamlit utilizará para crear formularios dinámicos.
config_modelo = {
    "modelo": "Random Forest",
    "target": TARGET,
    "features": FEATURES,
    "run_id": mejor_run_id,
    "periodicidad": "trimestral",
    "datos": "simulados",
    "version_config": 1
}
with open(carpeta_salidas / "config_modelo.json", "w", encoding="utf-8") as archivo:
    json.dump(config_modelo, archivo, indent=2, ensure_ascii=False)
print("Configuración del modelo exportada:", config_modelo)


### Entrenamiento final y exportación

El modelo se reentrena con toda la historia observada. **Suba el archivo `modelo_pib.joblib` junto con `config_modelo.json` y los CSV de `salidas_tracker/` al mismo repositorio**. La configuración se deriva de la selección real de predictores y no se debe modificar por separado.


In [ ]:
# Reentrenamos el mejor pipeline con TODA la historia observada
pipeline_rf = clone(modelo_ganador)
pipeline_rf.fit(X, y)

print("Modelo final entrenado correctamente")
print("Run ganador:", mejor_run_id)
print("Observaciones utilizadas:", len(X))
print("Variables utilizadas:", FEATURES)


In [ ]:
import joblib

# Guardamos el pipeline completo
joblib.dump(
    pipeline_rf,
    "modelo_pib.joblib"
)

print("Pipeline guardado correctamente")

In [ ]:
from google.colab import files

files.download("modelo_pib.joblib")

In [ ]:
import sys
import sklearn
import numpy
import pandas
import joblib

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("joblib:", joblib.__version__)